| # |  |  |
|---|----------|---------------|
| 1 | Regras de matching semânticas no prompt | Peeters & Bizer (2023) — regras em linguagem natural alcançam +5.94% F1 com custo mínimo |
| 3 | Pré-normalização do título antes do LLM | Ditto (Li et al., 2020) — span normalization reduz carga cognitiva do modelo |
| 4 | Batch prompting (múltiplos títulos por chamada) | BATCHER (Fan et al., 2024) — 4-7× redução de custo com qualidade equivalente |

**Pré-requisito:** `attack_augmentation.ipynb` → `data/attack.csv`

**Arquitetura (este notebook):**
1. Pré-Normalização Simbólica do Título (Ditto-inspired)
2. Extração via LLM com Batch Prompting
3. Limpeza Determinística (Camada Simbólica)
4. Exportação + relatório de robustez (`eval.py`)

**Benchmarks:** `benchmark.ipynb` → `output/normalized.json`, `output/by_attack_type.json`

## Setup

In [1]:
!pip install --upgrade pip
%pip install -q pandas numpy tqdm google-generativeai python-dotenv

   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------- ----------- 1.3/1.8 MB 7.4 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 5.2 MB/s  0:00:00



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: To modify pip, please run the following command:
c:\Users\nunes\.pyenv\pyenv-win\versions\3.12.3\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: c:\Users\nunes\.pyenv\pyenv-win\versions\3.12.3\python.exe -m pip install --upgrade pip


In [197]:
import os
import re
import json
import time
import logging
import pandas as pd
import numpy as np
from tqdm import tqdm
from pathlib import Path
from dotenv import load_dotenv
import google.generativeai as genai
import sys

# Carrega .env da pasta do notebook ou da raiz do repo (override=True evita
# chave inválida herdada do shell/Jupyter sobrescrever o .env).
for _env_dir in (Path.cwd(), Path.cwd().parent):
    _env_file = _env_dir / ".env"
    if _env_file.is_file():
        load_dotenv(_env_file, override=True)
        break
else:
    load_dotenv(override=True)

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

In [199]:
# ── Configuração ──────────────────────────────────────────────
ATTACK_PATH         = "data/attack.csv"      # Entrada (gerado por attack_augmentation.ipynb)
NORMALIZED_PATH     = "output/normalized.csv"

BATCH_SIZE          = 5
API_KEY             = (os.getenv("GOOGLE_API_KEY") or "").strip()
NORMALIZATION_MODEL = os.getenv("MODEL_NAME", "gemini-3.1-flash-lite-preview")
TEMPERATURE         = 0.0

# ── Execucoes repetidas (estabilidade probabilistica) ───────────
# RUN_ID=1 (padrao) usa os caminhos canonicos ja usados no artigo (sem sufixo).
# Para rodar de novo sem sobrescrever, defina RUN_ID=2,3,4,5 (env var ou
# diretamente aqui) antes de re-executar as secoes dos Passos 6, 9 e 10.
RUN_ID     = os.getenv("RUN_ID", "3").strip()
RUN_SUFFIX = "" if RUN_ID == "1" else f"_run{RUN_ID}"
print(f"RUN_ID={RUN_ID!r}  RUN_SUFFIX={RUN_SUFFIX!r}")


RUN_ID='3'  RUN_SUFFIX='_run3'


## Passo 1 — Pré-Normalização do Título Atacado (Ditto-inspired)

> Aplica regras simbólicas nos títulos **atacados** antes do LLM de normalização,
> reduzindo variação na entrada (Ditto, Li et al., 2020).

In [173]:
# ── Pré-normalização do título (Ditto-inspired) ───────────────
# Aplicada ANTES do LLM para reduzir variação na entrada.
# A camada simbólica PÓS-LLM continua existindo como safety net.
#
# Princípio: regras CONSERVADORAS — preferir não-normalizar a normalizar
# errado. Bug histórico: o `k` solto matcheava "4K" em "Smart TV 4K",
# produzindo "Smart TV 4kg". A regra de quilos agora exige a palavra
# completa (kg/kilo/kilograma) para evitar essa classe de regressão.
# O mesmo cuidado vale para gramas (não pode pegar "5G" / "8GB").

_PRE_RULES = [
    # ── 1. Frações de litro textuais → ml (mais específico primeiro) ──
    (re.compile(r"\b1/2\s*l(?:itro)?s?\b", re.I),         "500ml"),
    (re.compile(r"\bmeio\s+litro", re.I),                  "500ml"),

    # ── 2. Volume em litros → ml ──
    # Aceita inteiro ou decimal (vírgula ou ponto). Ex: 0,9L / 1.5l / 2 litros
    (re.compile(r"(\d+(?:[,.]\d+)?)\s*l(?:itro[s]?)?\b", re.I),
        lambda m: (
            f"{int(float(m.group(1).replace(',', '.')) * 1000)}ml"
            if float(m.group(1).replace(',', '.')) * 1000
               == int(float(m.group(1).replace(',', '.')) * 1000)
            else f"{float(m.group(1).replace(',', '.')) * 1000}ml"
        )),

    # ── 3. Mililitros → padronização sem espaço ──
    (re.compile(r"(\d+(?:[,.]\d+)?)\s*ml\b", re.I),
        lambda m: f"{m.group(1).replace(',', '.')}ml"),

    # ── 4. Quilos → kg ──
    # ATENÇÃO: exige palavra completa (kg, kilo, kilograma).  Caso contrário
    # consume "4K"/"8K" (resoluções de TV).  Regressão histórica documentada.
    (re.compile(r"(\d+(?:[,.]\d+)?)\s*(?:kg|kilos?|kilogramas?)\b", re.I),
        lambda m: f"{m.group(1).replace(',', '.')}kg"),

    # ── 5. Gramas → g ──
    # Exige "gramas?" ou "gr" para não colidir com "5G" (rede móvel) ou "8GB".
    (re.compile(r"(\d+(?:[,.]\d+)?)\s*(?:gramas?|gr)\b", re.I),
        lambda m: f"{m.group(1).replace(',', '.')}g"),

    # ── 6. Voltagem ──
    (re.compile(r"\bbi[\s-]?volt(?:s)?\b", re.I),         "bivolt"),
    # Aceita "110v", "110V", "110volt", "110volts" (regra anterior exigia "vol").
    (re.compile(r"(\d+)\s*v(?:olts?)?\b", re.I),
        lambda m: f"{m.group(1)}v"),

    # ── 7. Watts → padronização sem espaço ──
    # Cobre "1000w", "1000 W", "1000 watt", "1000 watts".
    (re.compile(r"(\d+(?:[,.]\d+)?)\s*w(?:atts?)?\b", re.I),
        lambda m: f"{m.group(1).replace(',', '.')}w"),

    # ── 8. Polegadas → "Xpol" ──
    # Cobre `55"`, `55 polegadas`, `55 pol.` (display de TVs e form-factors).
    (re.compile(r'(\d+(?:[,.]\d+)?)\s*(?:"|polegadas?|pol\.?)(?=\s|$|[^a-z0-9])', re.I),
        lambda m: f"{m.group(1).replace(',', '.')}pol"),

    # ── 9. Espaços excessivos ──
    (re.compile(r"\s{2,}"), " "),
]


def pre_normalize_title(title: str) -> str:
    """Aplica regras de normalização no título ANTES de enviar ao LLM."""
    text = title.strip()
    for pattern, replacement in _PRE_RULES:
        text = pattern.sub(replacement, text)
    return text.strip()


# ── Teste rápido (inclui regressões conhecidas) ──
test_cases = [
    "Maquina De Lavar Electrolux 12 Kilos",
    "Garrafa Térmica 1/2 Litro Aço Inox",
    "Liquidificador Mondial Turbo 110 Volts",
    "Coca Cola meio litro",
    "Panela de Pressão 4,5 Litros",
    "Cerveja Heineken 600 ml Pack 6",
    # Regressões fixadas:
    "Smart TV Samsung 55 4K Crystal Bivolt",   # 4K NÃO pode virar 4kg
    "Smart TV LG 65 polegadas 8K",              # 8K preservado, 65 → 65pol
    "Liq. Philco 1000 Watts 110V",              # watts e 110V normalizados
    "Memória 8GB DDR4",                          # 8GB preservado
    "Roteador 5G Modem",                         # 5G preservado
    'TV 50" 4K HDR',                             # aspas → pol
]
print("Pré-normalização (Ditto-inspired):")
print("-" * 65)
for t in test_cases:
    print(f"  {t:<50s} → {pre_normalize_title(t)}")


Pré-normalização (Ditto-inspired):
-----------------------------------------------------------------
  Maquina De Lavar Electrolux 12 Kilos               → Maquina De Lavar Electrolux 12kg
  Garrafa Térmica 1/2 Litro Aço Inox                 → Garrafa Térmica 500ml Aço Inox
  Liquidificador Mondial Turbo 110 Volts             → Liquidificador Mondial Turbo 110v
  Coca Cola meio litro                               → Coca Cola 500ml
  Panela de Pressão 4,5 Litros                       → Panela de Pressão 4500ml
  Cerveja Heineken 600 ml Pack 6                     → Cerveja Heineken 600ml Pack 6
  Smart TV Samsung 55 4K Crystal Bivolt              → Smart TV Samsung 55 4K Crystal bivolt
  Smart TV LG 65 polegadas 8K                        → Smart TV LG 65pol 8K
  Liq. Philco 1000 Watts 110V                        → Liq. Philco 1000w 110v
  Memória 8GB DDR4                                   → Memória 8GB DDR4
  Roteador 5G Modem                                  → Roteador 5G Modem
  TV 50"

In [174]:
_attack_path = Path(ATTACK_PATH)
if _attack_path.is_file():
    df_attack = pd.read_csv(ATTACK_PATH)
    print(f"attack.csv carregado: {len(df_attack)} linhas")
else:
    df_attack = None
    print(f"⚠️  {ATTACK_PATH} não encontrado — pipeline de ataque (Passos 1-5) será pulado. Passo 6 (pairs_to_label) funciona normalmente.")

⚠️  data/attack.csv não encontrado — pipeline de ataque (Passos 1-5) será pulado. Passo 6 (pairs_to_label) funciona normalmente.


In [175]:
if df_attack is not None:
    df_attack["titulo_pre_normalizado"] = df_attack["attacked"].apply(pre_normalize_title)

    alterados = (df_attack["attacked"] != df_attack["titulo_pre_normalizado"]).sum()
    print(f"Títulos alterados pela pré-normalização: {alterados}/{len(df_attack)} ({alterados/len(df_attack)*100:.1f}%)")
    df_attack[df_attack["attacked"] != df_attack["titulo_pre_normalizado"]][["attacked", "titulo_pre_normalizado"]].head(10)

## Passo 2 — Filtro de Distância de Edição Ponderada (Pré-LLM)

Calcula uma Levenshtein ponderada entre `original` e `attacked` (ambos pré-normalizados, para que conversões de unidade tipo `0,5L` → `500ml` não sejam falsamente rejeitadas). Custos por tipo de caractere:

| Operação | Custo | Motivação |
|---|---|---|
| letra ↔ letra (sub) | 0.1 | barato — typo/grafia |
| letra inserir/deletar | 0.2 | barato — abreviações (`Liquidificador` → `Liq.`) |
| dígito ↔ dígito (sub) | 10.0 | proibitivo — `1L` vs `2L` = produtos diferentes |
| dígito inserir/deletar | 8.0 | alto — quantidade fantasma |
| outros (punct, espaço, mistos) | 0.3 | intermediário |

A distância bruta é normalizada por `max(len(s1), len(s2))`. Pares com distância acima de `EDIT_DISTANCE_THRESHOLD` (default `0.25`) são descartados antes do LLM de normalização, com log do motivo.

**Limitação intencional:** diferenças semânticas sem dígitos (ex.: `INTEGRAL` vs `DESNATADO`) ficam com distância baixa e seguem para o LLM — é ele quem resolve essa camada. Trocas de um único dígito em strings muito longas (>40 chars) podem também passar pelo filtro: o LLM/benchmark capta a divergência depois.

In [176]:
# ── Distância de edição ponderada (pré-filtragem) ─────────────
# Penaliza fortemente diferenças em dígitos: "1L" vs "2L" são produtos
# distintos mesmo com 1 caractere diferente.  Mantém custo baixo para
# variações ortográficas e abreviações — o LLM resolve essa camada.

EDIT_DISTANCE_THRESHOLD = 0.25

_COST_LETTER_SUB   = 0.1   # letra ↔ letra (substituição) — barato
_COST_LETTER_INDEL = 0.2   # letra inserir/deletar — barato (cobre abreviações)
_COST_DIGIT_SUB    = 10.0  # qualquer sub envolvendo dígito — proibitivo
_COST_DIGIT_INDEL  = 8.0   # dígito inserir/deletar — alto (quantidade fantasma)
_COST_OTHER        = 0.3   # punct/espaço (sub e indel sem dígito) — intermediário


def _indel_cost(c: str) -> float:
    if c.isdigit():
        return _COST_DIGIT_INDEL
    if c.isalpha():
        return _COST_LETTER_INDEL
    return _COST_OTHER


def _sub_cost(a: str, b: str) -> float:
    if a == b:
        return 0.0
    # Qualquer substituição envolvendo dígito é cara — caso contrário o DP
    # acha caminhos baratos do tipo "insert dígito + sub dígito↔letra (other)"
    # que burlam a penalidade de dígito↔dígito.
    if a.isdigit() or b.isdigit():
        return _COST_DIGIT_SUB
    if a.isalpha() and b.isalpha():
        return _COST_LETTER_SUB
    return _COST_OTHER


def weighted_edit_distance(s1: str, s2: str) -> float:
    """Levenshtein ponderada por tipo de caractere. Case-insensitive."""
    s1, s2 = s1.lower(), s2.lower()
    m, n = len(s1), len(s2)
    if m == 0:
        return sum(_indel_cost(c) for c in s2)
    if n == 0:
        return sum(_indel_cost(c) for c in s1)

    prev = [0.0] * (n + 1)
    for j in range(1, n + 1):
        prev[j] = prev[j - 1] + _indel_cost(s2[j - 1])

    for i in range(1, m + 1):
        curr = [prev[0] + _indel_cost(s1[i - 1])]
        for j in range(1, n + 1):
            curr.append(min(
                curr[j - 1] + _indel_cost(s2[j - 1]),             # insert
                prev[j]     + _indel_cost(s1[i - 1]),             # delete
                prev[j - 1] + _sub_cost(s1[i - 1], s2[j - 1]),    # substitute
            ))
        prev = curr
    return prev[n]


def normalized_weighted_distance(s1: str, s2: str) -> float:
    """Distância ponderada normalizada por max(len(s1), len(s2)). 0.0 se ambos vazios."""
    denom = max(len(s1), len(s2))
    if denom == 0:
        return 0.0
    return weighted_edit_distance(s1, s2) / denom


# ── Sanity checks (espec do enunciado) ───────────────────────
_checks = [
    ("1L coca",            "2L coca",              "DROP"),  # dígito trocado
    ("500ml leite",        "750ml leite",          "DROP"),  # dígito
    ("12kg",               "13kg",                 "DROP"),  # peso
    ("110v",               "220v",                 "DROP"),  # voltagem
    ("Coca-Cola 500ml",    "Coca-Cola 750ml",      "DROP"),  # dígito em string longa
    ("Smart TV 4K",        "Smart TV 8K",          "DROP"),  # resolução
    ("Coca-Cola Lata",     "Coca-Cola Lata 350ml", "DROP"),  # qty adicionada
    ("leite integral",     "leite desnatado",      "KEEP"),  # semântico, sem dígito
    ("Liquidificador",     "Liq.",                 "KEEP"),  # abreviação
    ("Coca-Cola 500ml",    "CocaCola 500ml",       "KEEP"),  # só pontuação
    ("Samsung Galaxy",     "Samsugn Galaxy",       "KEEP"),  # typo
    ("500ml",              "500ml",                "KEEP"),  # idêntico
    ("Notebook Dell",      "Notebook Lenovo",      "KEEP"),  # marca, semântico
]
print(f"EDIT_DISTANCE_THRESHOLD = {EDIT_DISTANCE_THRESHOLD}")
print("-" * 72)
for a, b, expected in _checks:
    d = normalized_weighted_distance(a, b)
    verdict = "DROP" if d > EDIT_DISTANCE_THRESHOLD else "KEEP"
    ok = "OK " if verdict == expected else "!! "
    print(f"  {ok}{verdict}  d={d:.3f}  | {a!r}  vs  {b!r}")


EDIT_DISTANCE_THRESHOLD = 0.25
------------------------------------------------------------------------
  OK DROP  d=1.429  | '1L coca'  vs  '2L coca'
  OK DROP  d=1.455  | '500ml leite'  vs  '750ml leite'
  OK DROP  d=2.500  | '12kg'  vs  '13kg'
  OK DROP  d=5.000  | '110v'  vs  '220v'
  OK DROP  d=1.067  | 'Coca-Cola 500ml'  vs  'Coca-Cola 750ml'
  OK DROP  d=0.909  | 'Smart TV 4K'  vs  'Smart TV 8K'
  OK DROP  d=1.235  | 'Coca-Cola Lata'  vs  'Coca-Cola Lata 350ml'
  OK KEEP  d=0.060  | 'leite integral'  vs  'leite desnatado'
  OK KEEP  d=0.164  | 'Liquidificador'  vs  'Liq.'
  OK KEEP  d=0.020  | 'Coca-Cola 500ml'  vs  'CocaCola 500ml'
  OK KEEP  d=0.014  | 'Samsung Galaxy'  vs  'Samsugn Galaxy'
  OK KEEP  d=0.000  | '500ml'  vs  '500ml'
  OK KEEP  d=0.047  | 'Notebook Dell'  vs  'Notebook Lenovo'


In [177]:
if df_attack is not None:
    df_attack["category"] = ""

In [178]:
if df_attack is not None:
    df_attack["original_pre_normalizado"] = df_attack["original"].apply(pre_normalize_title)
    df_attack["edit_distance"] = df_attack.apply(
        lambda r: normalized_weighted_distance(
            r["original_pre_normalizado"], r["titulo_pre_normalizado"]
        ),
        axis=1,
    )

    mask_keep = df_attack["edit_distance"] <= EDIT_DISTANCE_THRESHOLD
    n_total = len(df_attack)
    n_drop  = int((~mask_keep).sum())

    if n_drop:
        logger.info(
            "Filtro de distância: %d/%d pares descartados (%.1f%%) — threshold=%.2f",
            n_drop, n_total, n_drop / n_total * 100, EDIT_DISTANCE_THRESHOLD,
        )
        df_dropped = df_attack.loc[~mask_keep, ["original", "attacked", "tipo_variacao", "edit_distance"]]
        for row in df_dropped.itertuples(index=False):
            logger.info(
                "  DROP  d=%.3f  [%s]  %r -> %r",
                row.edit_distance, row.tipo_variacao, row.original, row.attacked,
            )
    else:
        logger.info("Filtro de distância: nenhum par descartado (%d mantidos)", n_total)

    df_attack = df_attack.loc[mask_keep].reset_index(drop=True)

    if n_drop and "tipo_variacao" in df_dropped.columns:
        print("Descartes por tipo de variação:")
        print(df_dropped["tipo_variacao"].value_counts().to_string())
    print(f"\nPares mantidos: {len(df_attack)}/{n_total}")

## Passo 3 — Normalização via LLM (Batch)

### Regras de Matching Semânticas (Peeters & Bizer, 2023)
> Regras em linguagem natural no prompt alcançaram 88.29% F1 (+5.94%).

### Batch Prompting (BATCHER, Fan et al., 2024)
> 4-7× redução de custo de API com qualidade equivalente.

In [179]:
SYSTEM_PROMPT = """
Você é um especialista em extração e padronização (normalização) estrita de atributos de produtos de e-commerce brasileiro. A sua saída será usada para comparar se dois títulos representam EXACTAMENTE o mesmo produto no mundo real.

══════════════════════════════════════════════════════════════════
TAREFA
══════════════════════════════════════════════════════════════════
Você receberá um ou mais títulos de produtos. Para CADA título, extraia APENAS dados técnicos e reais. Ignore textos de marketing, condições de uso ou contexto. 

Chaves obrigatórias (use null se o atributo não existir no título):
  "idx"                : int    — o índice recebido
  "marca"              : string — ex: "Electrolux", "Samsung", "Tramontina"
  "modelo"             : string — ex: "LTD12", "Galaxy S21", null
  "tipo_produto"       : string — ex: "máquina de lavar", "geladeira", "furadeira". SEMPRE resuma a 1 a 3 palavras no máximo.
  "capacidade_volume"  : string — ex: "12kg", "500ml", "2000ml", null
  "voltagem"           : string — "110v", "220v" ou "bivolt"; null se ausente
  "cor"                : string — ex: "branco", "preto", null
  "material"           : string — ex: "inox", "plástico", null
  "quantidade"         : string — ex: "kit 3", "1", null (Se for pacote, extraia só o número)
  "outros_atributos"   : string — null para a maioria. Apenas se tiver uma ESPECIFICAÇÃO DE HARDWARE (ex: "classe 10", "DDR4", "21v"). NÃO coloque adjetivos aqui.

══════════════════════════════════════════════════════════════════
REGRAS DE NORMALIZAÇÃO DE VALORES
══════════════════════════════════════════════════════════════════
1. Volume → converter TODOS para ml sem espaço: "1/2 litro", "0,5 litro" → "500ml" | "1l" → "1000ml"
2. Voltagem → "110 v", "110 volts" → "110v" | "bi-volt" → "bivolt"
3. Peso/Capacidade → sem espaço: "12 kilos", "12 kg" → "12kg" | "500 gr" → "500g"
4. Minúsculo: Texto em minúsculo para todos os valores, exceto marcas.

══════════════════════════════════════════════════════════════════
REGRAS CRÍTICAS DE LIMPEZA (FILTRO DE LIXO)
══════════════════════════════════════════════════════════════════
5. OMITA ABSOLUTAMENTE adjetivos de marketing, texto promocional ou menção a público-alvo e eventos.
   - IGNORE palavras como: "gamer", "profissional", "dia das mães", "promoção", "original", "uso diário", "premium".
6. OMITA conectivos ("de", "para", "e", "com") e mantenha a essência direta.
   - Errado: "pote de vidro" -> Certo: "pote vidro"
   - Errado: "capa para celular" -> Certo: "capa celular"
   - Errado: "furadeira e lixadeira" -> Certo: "furadeira lixadeira"
7. DESCRIÇÕES REPETITIVAS: Se o título usa duas palavras quase sinônimas para a mesma coisa, simplifique para a mais comum.
   - Errado: "tênis sapatilha" -> Certo: "tênis"

══════════════════════════════════════════════════════════════════
REGRAS DE DESAMBIGUAÇÃO SEMÂNTICA
══════════════════════════════════════════════════════════════════
8. SINÔNIMOS DE TIPO (use a forma genérica curta ao invés de nomes longos ou gírias da área):
   - "lavadora" = "máquina de lavar" → "máquina lavar"
   - "refrigerador" = "geladeira"
   - "tv" = "televisão" = "smart tv" → "televisão"
   - "pendrive ultra slim" = "pen drive flash" → "pen drive"
   - "ração gatos castrados" = "comida gato adulto" → "ração para gato" (simplifique o descritivo de utilidade)

9. MARCA — unifique variações: "Coca Cola" = "coca cola" = "Coca-Cola" → "Coca-Cola".
   NÃO invente marcas. Se não tiver certeza se uma palavra é a fabricante oficial, prefira "null".

══════════════════════════════════════════════════════════════════
FORMATO DE SAÍDA (BATCH)
══════════════════════════════════════════════════════════════════
Retorne SOMENTE um array JSON válido. Sem texto extra, sem markdown.
"""

### Função de extração — modo batch

A função agora recebe uma lista de títulos e retorna uma lista de JSONs,
enviando todos em uma única chamada de API.

In [180]:
def extract_batch_llm(
    titles: list[str],
    categories: list[str],
    client: genai.GenerativeModel,
    start_idx: int = 0,
) -> list[dict]:
    """
    Envia múltiplos títulos em uma chamada (BATCHER-inspired).
    Inclui categoria como contexto (Narayan et al., 2022 — attribute selection).
    """
    lines = []
    for i, (title, cat) in enumerate(zip(titles, categories)):
        lines.append(f'[{start_idx + i}] Categoria: "{cat}" | Título: "{title}"')
    user_prompt = "\n".join(lines)

    try:
        response = client.generate_content(user_prompt)
        usage = response.usage_metadata
        raw_text = response.text.strip()

        # Limpa markdown se houver
        if raw_text.startswith("```"):
            raw_text = re.sub(r"^```(?:json)?", "", raw_text).rstrip("`").strip()

        parsed = json.loads(raw_text)

        # Se o LLM retornou um dict único em vez de array, embrulhar
        if isinstance(parsed, dict):
            parsed = [parsed]

        return parsed

    except json.JSONDecodeError:
        # Fallback: tentar extrair cada {...} individualmente
        matches = re.findall(r"\{[^{}]+\}", raw_text, re.DOTALL)
        results = []
        for m in matches:
            try:
                results.append(json.loads(m))
            except json.JSONDecodeError:
                continue
        if results:
            return results

        logger.warning("Sem JSON extraível do batch. Texto: %s", repr(raw_text[:200]))
        return [{} for _ in titles]

    except ValueError as e:
        logger.warning("Resposta bloqueada para batch: %s", e)
        return [{} for _ in titles]

    except Exception as e:
        logger.error("Erro LLM batch: %s", e)
        return [{} for _ in titles]

## Passo 4 — Limpeza Determinística (Camada Simbólica)

> Mantida como safety net pós-LLM.

In [181]:
_RE_SPACES     = re.compile(r"\s+")
_RE_VOLUME_ML  = re.compile(r"(\d+(?:[.,]\d+)?)\s*ml", re.I)
_RE_VOLUME_L   = re.compile(r"(\d+(?:[.,]\d+)?)\s*l(?:itro[s]?)?(?=\b|$)", re.I)
_RE_WEIGHT_KG  = re.compile(r"(\d+(?:[.,]\d+)?)\s*k(?:g|ilos?|ilogramas?)", re.I)
_RE_WEIGHT_G   = re.compile(r"(\d+(?:[.,]\d+)?)\s*g(?:r|ramas?)?(?=\b|$)", re.I)
_RE_VOLTAGE    = re.compile(r"(\d+)\s*v(?:olts?)?(?=\b|$)", re.I)
_RE_BIVOLT     = re.compile(r"bi[\s-]?volt", re.I)


def _clean_str(value) -> str:
    if not value or not isinstance(value, str):
        return ""
    return _RE_SPACES.sub(" ", value.strip()).lower()


def _normalize_volume(value: str) -> str:
    if not value:
        return value
    value = value.replace(",", ".")
    m = _RE_VOLUME_ML.search(value)
    if m:
        qty = float(m.group(1))
        return f"{int(qty)}ml" if qty == int(qty) else f"{qty}ml"
    m = _RE_VOLUME_L.search(value)
    if m:
        return f"{int(float(m.group(1)) * 1000)}ml"
    return value


def _normalize_weight(value: str) -> str:
    if not value:
        return value
    value = value.replace(",", ".")
    m = _RE_WEIGHT_KG.search(value)
    if m:
        qty = float(m.group(1))
        return f"{int(qty)}kg" if qty == int(qty) else f"{qty}kg"
    m = _RE_WEIGHT_G.search(value)
    if m:
        qty = float(m.group(1))
        return f"{int(qty)}g" if qty == int(qty) else f"{qty}g"
    return value


def _normalize_voltage(value: str) -> str:
    if not value:
        return value
    if _RE_BIVOLT.search(value):
        return "bivolt"
    m = _RE_VOLTAGE.search(value)
    return f"{m.group(1)}v" if m else value


def _normalize_quantidade(value: str):
    """Unidade unica (1, 01, "- 1") vira None -- sem isso a LLM as vezes
    extrai null e as vezes "1" pro mesmo caso implicito (nenhuma quantidade
    mencionada no titulo), gerando instabilidade entre execucoes no nome
    canonico ("produto" vs "produto - 1") sem ganho de informacao real.
    """
    if not value:
        return None
    m = re.search(r"\d+", str(value))
    if m and int(m.group()) == 1:
        return None
    return value


def _normalize_capacity(value: str) -> str:
    if not value:
        return value
    v = value.lower()
    if any(u in v for u in ["ml", "litro", " l", "l "]):
        return _normalize_volume(v)
    if any(u in v for u in ["kg", "kilo", "grama", " g", "g "]):
        return _normalize_weight(v)
    return _clean_str(value)


def clean_llm_output(raw_json: dict) -> dict:
    if not raw_json:
        return {}
    cleaned = {}
    for key, value in raw_json.items():
        if key == "idx":
            continue
        if value is None or value == "null":
            cleaned[key] = None
            continue
        s = str(value).strip()
        if key == "marca":
            cleaned[key] = _RE_SPACES.sub(" ", s).title() or None
        elif key == "voltagem":
            cleaned[key] = _normalize_voltage(_clean_str(s)) or None
        elif key == "capacidade_volume":
            cleaned[key] = _normalize_capacity(_clean_str(s)) or None
        elif key == "quantidade":
            cleaned[key] = _normalize_quantidade(_clean_str(s))
        else:
            cleaned[key] = _clean_str(s) or None
    return cleaned


# ── Match key (estável) vs nome canônico (display) ────────────
# A constancy do benchmark é sensível a QUALQUER divergência entre
# variantes do mesmo `original`.  Ataques adversariais frequentemente
# adicionam ou removem atributos secundários (cor, sedan/0km,
# outros_atributos), o que estoura a métrica mesmo quando o produto
# é o mesmo.  Por isso geramos duas formas:
#
#   • match_key   — só atributos estáveis (tipo, marca, modelo,
#                   capacidade, voltagem). Consumido pelo benchmark
#                   na coluna `normalized` do CSV.
#   • canonical   — string completa para inspeção humana, salva
#                   na coluna `display_name` do CSV.
#
# Caso emblemático: "Nissan Versa 0km" vs "Nissan Versa Sedan" geram
# `outros_atributos` divergentes mas representam o mesmo produto.
# Match key colapsa as duas em `carro | Nissan | versa`.

_STABLE_FIELDS = (
    "tipo_produto",
    "marca",
    "modelo",
    "capacidade_volume",
    "voltagem",
)


def build_match_key(cleaned_json: dict) -> str:
    """Chave estável usada pelo benchmark de constância."""
    if not cleaned_json:
        return ""
    parts = [cleaned_json.get(f) for f in _STABLE_FIELDS]
    return " | ".join(p for p in parts if p)


def build_canonical_name(cleaned_json: dict) -> str:
    """Formato display: ${nome} ${marca} ${cor} ${outros} - ${quantidade} ${capacidade}"""
    nome  = cleaned_json.get("tipo_produto") or ""
    marca = cleaned_json.get("marca") or ""
    cor   = cleaned_json.get("cor") or ""

    outros_parts = [
        cleaned_json.get("modelo"),
        cleaned_json.get("material"),
        cleaned_json.get("voltagem"),
        cleaned_json.get("outros_atributos"),
    ]
    outros = ", ".join(p for p in outros_parts if p)

    cap = cleaned_json.get("capacidade_volume") or ""
    m_cap = re.match(r"(\d+(?:\.\d+)?)\s*([a-zA-Z]+)", cap)
    cap_str = f"{m_cap.group(1)} {m_cap.group(2)}" if m_cap else cap

    # quantidade era extraida pela LLM mas nunca chegava no nome final -- bug
    # que colapsava produtos com quantidades diferentes (ex.: "kit 12" e
    # "20 unidades") no mesmo nome canonico. Agora entra como componente
    # proprio, separado de capacidade_volume.
    qtd = cleaned_json.get("quantidade") or ""

    main = " ".join(p for p in [nome, marca, cor, outros] if p)
    tail = " ".join(p for p in [qtd, cap_str] if p)
    if tail:
        return f"{main} - {tail}".strip()
    return main

## Passo 5 — Orquestração e Exportação

In [182]:
os.makedirs("output", exist_ok=True)

genai.configure(api_key=API_KEY)
client = genai.GenerativeModel(
    model_name=NORMALIZATION_MODEL,
    system_instruction=SYSTEM_PROMPT,
    generation_config=genai.GenerationConfig(
        response_mime_type="application/json",
        temperature=TEMPERATURE,
        max_output_tokens=2048,
    ),
)

In [183]:
if df_attack is not None:
    titles_pre = df_attack["titulo_pre_normalizado"].tolist()
    n = len(titles_pre)

    all_raw_jsons   = [None] * n
    all_cleaned     = [None] * n
    all_canonical   = [None] * n
    all_match_keys  = [None] * n

    for batch_start in tqdm(range(0, n, BATCH_SIZE), desc="Normalizando", unit="batch"):
        batch_end    = min(batch_start + BATCH_SIZE, n)
        batch_titles = titles_pre[batch_start:batch_end]
        batch_cats   = [""] * len(batch_titles)

        raw_list = extract_batch_llm(batch_titles, batch_cats, client, start_idx=batch_start)

        for local_i in range(batch_end - batch_start):
            global_i = batch_start + local_i

            matched = None
            for r in raw_list:
                if isinstance(r, dict) and r.get("idx") == global_i:
                    matched = r
                    break
            if matched is None and local_i < len(raw_list):
                matched = raw_list[local_i] if isinstance(raw_list[local_i], dict) else {}
            if matched is None:
                matched = {}

            cleaned   = clean_llm_output(matched)
            canon     = build_canonical_name(cleaned)
            match_key = build_match_key(cleaned)

            all_raw_jsons[global_i]  = matched
            all_cleaned[global_i]    = cleaned
            all_canonical[global_i]  = canon
            all_match_keys[global_i] = match_key

        time.sleep(2)

    print(f"Normalização concluída: {n} títulos em {(n + BATCH_SIZE - 1) // BATCH_SIZE} batches")
else:
    print("Pipeline de ataque pulado (data/attack.csv não encontrado).")

Pipeline de ataque pulado (data/attack.csv não encontrado).


In [184]:
if df_attack is not None:
    # `normalized`   = display_name (nome canônico completo) — consumido pelo benchmark e pelo matching.
    # `match_key`    = chave estável com subconjunto de campos — mantida para referência.
    df_result = pd.DataFrame({
        "original":     df_attack["original"].values,
        "attacked":     df_attack["attacked"].values,
        "normalized":   all_canonical,
        "match_key":    all_match_keys,
    })

    df_result.to_csv(NORMALIZED_PATH, index=False, encoding="utf-8-sig")
    print(f"Exportado: {NORMALIZED_PATH}")
    df_result.head(10)
else:
    print("Pipeline de ataque pulado — output/normalized.csv não gerado.")

Pipeline de ataque pulado — output/normalized.csv não gerado.


In [185]:
if Path(NORMALIZED_PATH).is_file():
    df_result = pd.read_csv(NORMALIZED_PATH)
else:
    df_result = None
    print(f"output/normalized.csv não encontrado — seção de relatório será pulada.")

output/normalized.csv não encontrado — seção de relatório será pulada.


## Passo 6 — Normalização de `dados/pairs_to_label.csv`

Lê o CSV de pares rotulados, normaliza `title_a` e `title_b` via mesma pipeline (pré-normalização + LLM + limpeza simbólica), e exporta `dados/pairs_normalized.csv` com as colunas `item_a_normalized` e `item_b_normalized` (nome canônico completo, equivalente ao `display_name`).

**Pré-requisito:** as funções `pre_normalize_title`, `extract_batch_llm`, `clean_llm_output` e `build_canonical_name` devem estar definidas (células anteriores deste notebook).

In [186]:
import re as _re_pairs

PAIRS_INPUT_CSV  = "dados/pairs_to_label.csv"
PAIRS_OUTPUT_CSV = f"dados/pairs_normalized{RUN_SUFFIX}.csv"

_HASH_SUFFIX = _re_pairs.compile(r"\s*#[a-z0-9]{4,}\s*$", _re_pairs.I)


def _strip_hash(title: str) -> str:
    """Remove sufixo de hash como '#pdxx' que não faz parte do título real."""
    return _HASH_SUFFIX.sub("", title).strip()


pairs_input_path = Path(PAIRS_INPUT_CSV).resolve()
if not pairs_input_path.is_file():
    raise FileNotFoundError(f"CSV não encontrado: {pairs_input_path}")

df_pairs = pd.read_csv(pairs_input_path, dtype=str)
df_pairs["label"] = df_pairs["label"].str.strip()

print(f"Pares carregados: {len(df_pairs)}")
print(f"Distribuição de labels: {df_pairs['label'].value_counts().to_dict()}")

# Títulos brutos (A e B), sem hash
titles_a_clean = [_strip_hash(t) for t in df_pairs["title_a"].tolist()]
titles_b_clean = [_strip_hash(t) for t in df_pairs["title_b"].tolist()]

# Conjunto único de títulos para evitar chamadas LLM redundantes
unique_titles = list(dict.fromkeys(titles_a_clean + titles_b_clean))
print(f"\nTítulos únicos a normalizar: {len(unique_titles)}")

Pares carregados: 1200
Distribuição de labels: {'0': 1170, '1': 30}

Títulos únicos a normalizar: 2257


In [17]:
# Normaliza os títulos únicos via LLM (mesma pipeline do Passo 3/4/5)
unique_pre = [pre_normalize_title(t) for t in unique_titles]
n_unique = len(unique_pre)

pairs_canonical_list = [None] * n_unique
pairs_cleaned_list   = [None] * n_unique  # campos estruturados para comparação campo-a-campo

for batch_start in tqdm(range(0, n_unique, BATCH_SIZE), desc="Normalizando pares", unit="batch"):
    batch_end    = min(batch_start + BATCH_SIZE, n_unique)
    batch_titles = unique_pre[batch_start:batch_end]
    batch_cats   = [""] * len(batch_titles)

    raw_list = extract_batch_llm(batch_titles, batch_cats, client, start_idx=batch_start)

    for local_i in range(batch_end - batch_start):
        global_i = batch_start + local_i
        matched = None
        for r in raw_list:
            if isinstance(r, dict) and r.get("idx") == global_i:
                matched = r
                break
        if matched is None and local_i < len(raw_list):
            matched = raw_list[local_i] if isinstance(raw_list[local_i], dict) else {}
        if matched is None:
            matched = {}

        cleaned = clean_llm_output(matched)
        pairs_canonical_list[global_i] = build_canonical_name(cleaned)
        pairs_cleaned_list[global_i]   = cleaned  # salva dict de campos estruturados

    time.sleep(0.5)

title_to_norm    = {unique_titles[i]: pairs_canonical_list[i] or "" for i in range(n_unique)}
title_to_cleaned = {unique_titles[i]: pairs_cleaned_list[i]   or {} for i in range(n_unique)}
print(f"Normalização concluída: {n_unique} títulos únicos")

Normalizando pares:   0%|          | 0/452 [00:00<?, ?batch/s]

Normalizando pares:  96%|█████████▌| 432/452 [18:19<00:52,  2.63s/batch]2026-08-21 20:51:38,367 [WARNING] Resposta bloqueada para batch: Invalid operation: The `response.parts` quick accessor requires a single candidate, but but `response.candidates` is empty.
This appears to be caused by a blocked prompt, see `response.prompt_feedback`: block_reason: PROHIBITED_CONTENT

Normalizando pares: 100%|██████████| 452/452 [19:09<00:00,  2.54s/batch]

Normalização concluída: 2257 títulos únicos


In [18]:
df_pairs["item_a_normalized"] = [title_to_norm.get(t, "") for t in titles_a_clean]
df_pairs["item_b_normalized"] = [title_to_norm.get(t, "") for t in titles_b_clean]

# Campos estruturados — usados pela comparação campo-a-campo no benchmark
_STRUCT_FIELDS = ["tipo_produto", "marca", "modelo", "capacidade_volume", "voltagem", "quantidade"]
for _field in _STRUCT_FIELDS:
    df_pairs[f"a_{_field}"] = [title_to_cleaned.get(t, {}).get(_field) for t in titles_a_clean]
    df_pairs[f"b_{_field}"] = [title_to_cleaned.get(t, {}).get(_field) for t in titles_b_clean]

out_pairs_path = Path(PAIRS_OUTPUT_CSV).resolve()
df_pairs.to_csv(out_pairs_path, index=False, encoding="utf-8-sig")

print(f"Exportado: {out_pairs_path}")
print(f"Colunas: {list(df_pairs.columns)}")
df_pairs[["title_a", "title_b", "label", "item_a_normalized", "item_b_normalized"]].head(10)

Exportado: C:\Users\nunes\invoice-data-normalizer\main_supersaurus\dados\pairs_normalized_run3.csv
Colunas: ['title_a', 'title_b', 'candidate_label', 'similarity_score', 'label', 'item_a_normalized', 'item_b_normalized', 'a_tipo_produto', 'b_tipo_produto', 'a_marca', 'b_marca', 'a_modelo', 'b_modelo', 'a_capacidade_volume', 'b_capacidade_volume', 'a_voltagem', 'b_voltagem', 'a_quantidade', 'b_quantidade']


,title_a,title_b,label,item_a_normalized,item_b_normalized
0,Pulseira Casio Original Prata Gshock Dw5600e,Relógio Bvulgari,0,"pulseira relógio Casio prata dw5600e, g-shock",relógio Bvulgari
1,Trimpot 3386f De 500k Uma Volta Lote Com 25 Peças,Potenciometro Do A16o A190,0,"trimpot 3386f, 500k - 25",potenciômetro a160 a190
2,Relógio De Parede Para Cozinha Talher Colher P...,Relógio Parede Personalizado Lata Pokemon,0,relógio parede,"relógio parede pokemon, lata, personalizado"
3,Mesa Em Madeira Rústica-pubs-bar- Johnnie Walk...,Frete Grátis | Cerveja 55 Sem Glúten | 18 Lata...,0,"mesa Johnnie Walker red label, madeira",cerveja 55 sem glúten - 18 710 ml
4,Cadeirinha Reclinável Preta Maestro Bebê 0 À 2...,Chicco - Xpace - Cadeirinha P/ Auto - Excelent...,0,"cadeirinha bebê Multikids preta maestro, recli...",cadeirinha auto Chicco xpace
5,Nvr Ecd 8 Canais 4k Poe Lux Vision,Mangueira Reservatorio Radiador Jahu - Escort ...,0,"nvr Lux Vision ecd, 4k poe - 8","mangueira reservatório Jahu ja10807, escort zetec"
6,Discos Pastilhas Sensor Diant. - Tras. Bmw 320...,Disco Dianteiro Audi Q5 2.0 225cv Ambition 2015,0,"pastilha freio Bmw 320i 2.0 f30, sensor diante...","disco freio Audi q5, 2.0 225cv 2015"
7,Motu 8pre Placa De Som/interface De Audio Fire...,M-audio 2626 Profire Interface Digital De Grav...,0,"interface audio Motu 8pre, firewire","interface áudio M-Audio profire 2626, digital"
8,Mocassim Drive Infantil Selten Em Couro 003c,Sapato Social Bigioni Executivo Original Couro...,0,"mocassim Selten 003c, couro",sapato social Bigioni couro
9,"Canudo Inox 200 Reto 7,5 Mm + 50 Escova Metal ...",Conjunto De Canudos Inoxidáveis,0,"canudo inox, 7,5mm reto - 200",canudos inox


In [19]:
if df_result is not None:
    from eval import gerar_relatorio_robusto
    gerar_relatorio_robusto(df_result)
else:
    print("Relatório pulado — df_result não disponível.")

Relatório pulado — df_result não disponível.


In [22]:
asde

NameError: name 'asde' is not defined

---

## Passo 7 — Normalização via Llama 3.3 (Cerebras API)

Alternativa ao Gemini usando o modelo **Llama 3.3 70B** via [Cerebras](https://cloud.cerebras.ai) — gratuito, sem limite diário de tokens, inferência muito rápida (wafer-scale).

**Mesmo pipeline do Passo 6:** pré-normalização simbólica → LLM (Llama) → limpeza determinística → nome canônico.  
**Saída:** `dados/pairs_normalized_llama.csv` — mesma estrutura do `pairs_normalized.csv`, pronto para o `benchmark.ipynb`.

**Para rodar:**
1. Acesse https://cloud.cerebras.ai/platform/apikeys e crie uma API key
2. Adicione `CEREBRAS_API_KEY=csk_...` no arquivo `.env` do projeto
3. Execute as células abaixo na ordem

In [187]:
### 7.1 — Configuração Cerebras
import os, re, json, time, logging
from pathlib import Path
from openai import OpenAI as _GroqOpenAI

CEREBRAS_API_KEY = (os.getenv("CEREBRAS_API_KEY") or "").strip()

if not CEREBRAS_API_KEY:
    raise EnvironmentError(
        "CEREBRAS_API_KEY não encontrada.\n"
        "Acesse https://cloud.cerebras.ai/platform/apikeys, crie uma key\n"
        "e adicione CEREBRAS_API_KEY=csk_... no arquivo .env do projeto."
    )

groq_client = _GroqOpenAI(
    api_key  = CEREBRAS_API_KEY,
    base_url = "https://api.cerebras.ai/v1",
)

GROQ_MODEL             = "llama3.1-70b"   # modelo disponível no Cerebras free tier
PAIRS_INPUT_CSV_LLAMA  = "dados/pairs_to_label.csv"
PAIRS_OUTPUT_CSV_LLAMA = "dados/pairs_normalized_llama.csv"
GROQ_BATCH_SIZE        = 5
GROQ_SLEEP_SEC         = 1.0
GROQ_MAX_TOKENS        = 2048

print(f"Modelo      : {GROQ_MODEL}")
print(f"API key     : {CEREBRAS_API_KEY[:8]}{'*' * (len(CEREBRAS_API_KEY) - 8)}")
print(f"Tempo total : ~{452 * (GROQ_SLEEP_SEC + 1) / 60:.0f} min estimado")
print(f"Saída       : {PAIRS_OUTPUT_CSV_LLAMA}")

Modelo      : llama3.1-70b
API key     : csk-2hkx********************************************
Tempo total : ~15 min estimado
Saída       : dados/pairs_normalized_llama.csv


In [188]:
### 7.2 — Função de extração via Groq

def extract_batch_groq(titles: list[str], start_idx: int = 0) -> list[dict]:
    """
    Mesma interface de extract_batch_llm, mas usa Groq + Llama.
    Groq suporta o formato de array JSON desde que o prompt seja claro —
    usamos o mesmo SYSTEM_PROMPT do Gemini sem alteração.
    """
    lines = [f'[{start_idx + i}] Título: "{t}"' for i, t in enumerate(titles)]
    user_prompt = "\n".join(lines)

    try:
        response = groq_client.chat.completions.create(
            model       = GROQ_MODEL,
            messages    = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": user_prompt},
            ],
            temperature = 0.0,
            max_tokens  = GROQ_MAX_TOKENS,
        )
        _content = response.choices[0].message.content
        if _content is None:
            raise ValueError(f"content=None (finish_reason={response.choices[0].finish_reason})")
        raw_text = _content.strip()

        # Remove markdown code fences caso o modelo as adicione
        if raw_text.startswith("```"):
            raw_text = re.sub(r"^```(?:json)?", "", raw_text).rstrip("`").strip()

        parsed = json.loads(raw_text)

        # Normaliza: se o modelo embrulhou num dict ({"results": [...]})
        if isinstance(parsed, dict):
            for key in ("results", "items", "products", "data", "output"):
                if key in parsed and isinstance(parsed[key], list):
                    parsed = parsed[key]
                    break
            else:
                parsed = [parsed]

        return parsed if isinstance(parsed, list) else [parsed]

    except json.JSONDecodeError:
        # Fallback: extrai objetos {} individualmente
        matches = re.findall(r"\{[^{}]+\}", raw_text, re.DOTALL)
        results = []
        for m in matches:
            try:
                results.append(json.loads(m))
            except json.JSONDecodeError:
                continue
        if results:
            return results
        logging.warning("Groq: sem JSON extraível. Preview: %s", repr(raw_text[:200]))
        return [{} for _ in titles]

    except Exception as e:
        logging.error("Erro Groq batch (idx=%d): %s", start_idx, e)
        return [{} for _ in titles]


# ── Smoke test (1 título) ──────────────────────────────────────
_test = extract_batch_groq(["Liquidificador Mondial Turbo 1000W 110v"], start_idx=0)
print("Smoke test OK:", json.dumps(_test, ensure_ascii=False))

2026-08-22 02:38:23,665 [INFO] HTTP Request: POST https://api.cerebras.ai/v1/chat/completions "HTTP/1.1 404 Not Found"
2026-08-22 02:38:23,675 [ERROR] Erro Groq batch (idx=0): Error code: 404 - {'message': 'Model does not exist or you do not have access to it.', 'type': 'not_found_error', 'param': 'model', 'code': 'model_not_found'}


Smoke test OK: [{}]


---

## Passo 8 — Normalização via Llama 3.1 8B (Groq)

Modelo menor e mais rápido do Groq — limite diário de tokens mais alto que o 70B.  
**Saída:** `dados/pairs_normalized_llama8b.csv`

**Para rodar:** mesma `GROQ_API_KEY` do Passo 7. Execute 8.1 → 8.2 em ordem.

In [200]:
### 8.1 — Config + normalização (Groq llama-3.1-8b-instant)
from openai import OpenAI as _OAI
import os, re, json, time, logging
from pathlib import Path
from tqdm import tqdm

_GROQ_KEY_8B = (os.getenv("GROQ_API_KEY") or "").strip()
if not _GROQ_KEY_8B:
    raise EnvironmentError("GROQ_API_KEY não encontrada no .env")

_client_8b  = _OAI(api_key=_GROQ_KEY_8B, base_url="https://api.groq.com/openai/v1")
_model_8b   = "llama-3.1-8b-instant"
_out_8b     = "dados/pairs_normalized_llama8b.csv"
_sleep_8b   = 6.0   # 4s → ~15 req/min; modelo tem limite de tokens/min mais alto

# ── Smoke test ────────────────────────────────────────────────
def _call(client, model, titles, start_idx=0):
    lines = [f'[{start_idx+i}] Título: "{t}"' for i, t in enumerate(titles)]
    try:
        r = client.chat.completions.create(
            model=model,
            messages=[{"role":"system","content":SYSTEM_PROMPT},
                      {"role":"user","content":"\n".join(lines)}],
            temperature=0.0, max_completion_tokens=2048,  # igualado ao Gemini (era 4096, assimetria sem justificativa)
            # response_format=json_object REMOVIDO: gpt-5.4-mini, forcado em modo
            # objeto, parou de devolver lista de 5 itens (so 1 preenchido, 4 caiam
            # no fallback {} -- ~80% de extracao vazia). Volta a depender so da
            # instrucao textual do prompt, como o DeepSeek nao teve esse problema
            # mas o GPT teve, mantemos a assimetria aqui de proposito.
        )
        _content = r.choices[0].message.content
        if _content is None:
            raise ValueError(f"content=None (finish_reason={r.choices[0].finish_reason})")
        raw = _content.strip()
        if raw.startswith("```"):
            raw = re.sub(r"^```(?:json)?","",raw).rstrip("`").strip()
        parsed = json.loads(raw)
        if isinstance(parsed, dict):
            # response_format=json_object obriga objeto no topo; o modelo embrulha
            # o array numa chave qualquer -- pega por FORMATO (unica lista), nao
            # por nome fixo, pra nao depender de adivinhar "results"/"items"/etc.
            _list_vals = [v for v in parsed.values() if isinstance(v, list)]
            if len(_list_vals) == 1:
                parsed = _list_vals[0]
            elif len(_list_vals) > 1:
                parsed = max(_list_vals, key=len)
            else:
                parsed = [parsed]
        return parsed if isinstance(parsed, list) else [parsed]
    except json.JSONDecodeError:
        hits = re.findall(r"\{[^{}]+\}", raw, re.DOTALL)
        out = []
        for h in hits:
            try: out.append(json.loads(h))
            except: pass
        return out or [{} for _ in titles]
    except Exception as e:
        logging.error("Erro batch (idx=%d): %s", start_idx, e)
        return [{} for _ in titles]

_t = _call(_client_8b, _model_8b, ["Liquidificador Mondial 1000W 110v"])
print(f"Smoke test {_model_8b}:", json.dumps(_t, ensure_ascii=False))

2026-08-22 03:24:09,407 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 401 Unauthorized"
2026-08-22 03:24:09,412 [ERROR] Erro batch (idx=0): Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'expired_api_key'}}


Smoke test llama-3.1-8b-instant: [{}]


---

## Passo 9 — Normalização via DeepSeek V4 Flash

**Nota:** este cabeçalho dizia "gpt-oss-120b (Cerebras)" mas o código abaixo chama a API da DeepSeek (`DEEPSEEK_API_KEY`, `deepseek-v4-flash`) — corrigido. O arquivo `dados/pairs_normalized_cerebras120b.csv` já existente foi gerado por uma versão anterior desta célula, antes de ser reaproveitada para a DeepSeek; não há mais célula neste notebook que o regenere.

**Saída:** `dados/pairs_normalized_deepseek.csv` (ou `..._runN.csv` se `RUN_ID` != 1)

**Para rodar:** defina `DEEPSEEK_API_KEY` no `.env`. Execute 9.1 → 9.2 em ordem.

In [190]:
### 9.1 — Config + smoke test (DeepSeek V4 Flash, modo não-pensante)
from openai import OpenAI as _OAI
import os, re, json, time, logging
from pathlib import Path

_DS_KEY = (os.getenv("DEEPSEEK_API_KEY") or "").strip()
if not _DS_KEY:
    raise EnvironmentError("DEEPSEEK_API_KEY não encontrada no .env")

_client_cbr = _OAI(api_key=_DS_KEY, base_url="https://api.deepseek.com")
_model_cbr  = "deepseek-v4-flash"
_out_cbr    = f"dados/pairs_normalized_deepseek{RUN_SUFFIX}.csv"
_sleep_cbr  = 0.5
_extra_cbr  = {"thinking": {"type": "disabled"}}  # equivalente ao antigo deepseek-chat

_t = _call(_client_cbr, _model_cbr, ["Liquidificador Mondial 1000W 110v"])
print(f"Smoke test {_model_cbr}:", json.dumps(_t, ensure_ascii=False))

2026-08-22 02:38:30,883 [INFO] HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"


Smoke test deepseek-v4-flash: [{"idx": 0, "marca": "Mondial", "modelo": null, "tipo_produto": "liquidificador", "capacidade_volume": null, "voltagem": "110v", "cor": null, "material": null, "quantidade": null, "outros_atributos": "1000w"}]


In [102]:
### 9.2 — Loop + exportação
import pandas as pd
from tqdm import tqdm

def _call_ds(titles, start_idx=0):
    """Chama DeepSeek com thinking desabilitado via extra_body."""
    lines = [f'[{start_idx+i}] Título: "{t}"' for i, t in enumerate(titles)]
    try:
        r = _client_cbr.chat.completions.create(
            model=_model_cbr,
            messages=[{"role": "system", "content": SYSTEM_PROMPT},
                      {"role": "user",   "content": "\n".join(lines)}],
            temperature=0.0,
            max_completion_tokens=2048,  # igualado ao Gemini (era 1024, assimetria sem justificativa)
            response_format={"type": "json_object"},  # forca JSON valido, igual ao Gemini (response_mime_type)
            extra_body=_extra_cbr,
        )
        _content = r.choices[0].message.content
        if _content is None:
            raise ValueError(f"content=None (finish_reason={r.choices[0].finish_reason})")
        raw = _content.strip()
        if raw.startswith("```"):
            raw = re.sub(r"^```(?:json)?", "", raw).rstrip("`").strip()
        parsed = json.loads(raw)
        if isinstance(parsed, dict):
            # response_format=json_object obriga objeto no topo; o modelo embrulha
            # o array numa chave qualquer -- pega por FORMATO (unica lista), nao
            # por nome fixo, pra nao depender de adivinhar "results"/"items"/etc.
            _list_vals = [v for v in parsed.values() if isinstance(v, list)]
            if len(_list_vals) == 1:
                parsed = _list_vals[0]
            elif len(_list_vals) > 1:
                parsed = max(_list_vals, key=len)
            else:
                parsed = [parsed]
        return parsed if isinstance(parsed, list) else [parsed]
    except json.JSONDecodeError:
        hits = re.findall(r"\{[^{}]+\}", raw, re.DOTALL)
        out = []
        for h in hits:
            try: out.append(json.loads(h))
            except: pass
        return out or [{} for _ in titles]
    except Exception as e:
        logging.error("Erro DeepSeek batch (idx=%d): %s", start_idx, e)
        return [{} for _ in titles]

_dfc = pd.read_csv("dados/pairs_to_label.csv", dtype=str)
_dfc["label"] = _dfc["label"].str.strip()
_HASH2 = re.compile(r"\s*#[a-z0-9]{4,}\s*$", re.I)
_tac = [_HASH2.sub("", t).strip() for t in _dfc["title_a"]]
_tbc = [_HASH2.sub("", t).strip() for t in _dfc["title_b"]]
_uniqc = list(dict.fromkeys(_tac + _tbc))
_nc    = len(_uniqc)
_prec  = [pre_normalize_title(t) for t in _uniqc]

_canonc   = [None] * _nc
_cleanedc = [None] * _nc

for _bs in tqdm(range(0, _nc, 5), desc=f"DeepSeek/{_model_cbr}", unit="batch"):
    _be  = min(_bs + 5, _nc)
    _raw = _call_ds(_prec[_bs:_be], start_idx=_bs)
    for _li in range(_be - _bs):
        _gi = _bs + _li
        _m  = next((r for r in _raw if isinstance(r, dict) and r.get("idx") == _gi), None)
        if _m is None: _m = _raw[_li] if _li < len(_raw) and isinstance(_raw[_li], dict) else {}
        _c  = clean_llm_output(_m)
        _canonc[_gi]   = build_canonical_name(_c)
        _cleanedc[_gi] = _c
    time.sleep(_sleep_cbr)

_t2nc = {_uniqc[i]: _canonc[i] or ""   for i in range(_nc)}
_t2cc = {_uniqc[i]: _cleanedc[i] or {} for i in range(_nc)}

_dfc["item_a_normalized"] = [_t2nc.get(t, "") for t in _tac]
_dfc["item_b_normalized"] = [_t2nc.get(t, "") for t in _tbc]
for _f in ["tipo_produto", "marca", "modelo", "capacidade_volume", "voltagem", "quantidade"]:
    _dfc[f"a_{_f}"] = [_t2cc.get(t, {}).get(_f) for t in _tac]
    _dfc[f"b_{_f}"] = [_t2cc.get(t, {}).get(_f) for t in _tbc]

_dfc.to_csv(_out_cbr, index=False, encoding="utf-8-sig")
print(f"Salvo: {_out_cbr}")
print("Benchmark: mude PAIRS_NORMALIZED_CSV =", repr(_out_cbr))

DeepSeek/deepseek-v4-flash: 100%|██████████| 452/452 [27:16<00:00,  3.62s/batch]

Salvo: dados/pairs_normalized_deepseek_run3.csv
Benchmark: mude PAIRS_NORMALIZED_CSV = 'dados/pairs_normalized_deepseek_run3.csv'


---

## Passo 10 — Normalização via GPT-5.4-mini (OpenAI)

Equivalente ao `gemini-3.1-flash-lite-preview` no tier OpenAI — modelo rápido e de baixo custo.
**Saída:** `dados/pairs_normalized_openai_gpt5.4mini.csv` (ou `..._runN.csv` se `RUN_ID` != 1)

**Para rodar:**
1. Acesse https://platform.openai.com/api-keys e crie uma API key
2. Adicione `OPENAI_API_KEY=sk-...` no arquivo `.env` do projeto
3. Execute as células 10.1 e 10.2 em ordem

In [201]:
### 10.1 — Config + smoke test (OpenAI gpt-4o-mini)
from openai import OpenAI as _OAI
import os, re, json, time, logging
from pathlib import Path

_OAI_KEY = (os.getenv("OPENAI_API_KEY") or "").strip()
if not _OAI_KEY:
    raise EnvironmentError(
        "OPENAI_API_KEY nao encontrada.\n"
        "Acesse https://platform.openai.com/api-keys, crie uma key\n"
        "e adicione OPENAI_API_KEY=sk-... no arquivo .env do projeto."
    )

_client_oai  = _OAI(api_key=_OAI_KEY)  # base_url padrao: api.openai.com
_model_oai   = "gpt-5.4-mini"
_out_oai     = f"dados/pairs_normalized_openai_gpt5.4mini{RUN_SUFFIX}.csv"
_sleep_oai   = 0.5  # tier pago: gargalo e a latencia da API, nao o rate limit

# Reutiliza _call() definida na secao 8 (mesma interface OpenAI-compat)
_t = _call(_client_oai, _model_oai, ["Liquidificador Mondial 1000W 110v"])
print(f"Smoke test {_model_oai}:", json.dumps(_t, ensure_ascii=False))

2026-08-22 03:24:18,901 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Smoke test gpt-5.4-mini: [{"idx": 0, "marca": "Mondial", "modelo": null, "tipo_produto": "liquidificador", "capacidade_volume": null, "voltagem": "110v", "cor": null, "material": null, "quantidade": null, "outros_atributos": "1000w"}]


In [202]:
### 10.2 — Loop + exportacao
import pandas as pd
from tqdm import tqdm

_dfo = pd.read_csv("dados/pairs_to_label.csv", dtype=str)
_dfo["label"] = _dfo["label"].str.strip()
_HASH_OAI = re.compile(r"\s*#[a-z0-9]{4,}\s*$", re.I)
_tao = [_HASH_OAI.sub("", t).strip() for t in _dfo["title_a"]]
_tbo = [_HASH_OAI.sub("", t).strip() for t in _dfo["title_b"]]
_uniqo = list(dict.fromkeys(_tao + _tbo))
_no    = len(_uniqo)
_preo  = [pre_normalize_title(t) for t in _uniqo]

_canono   = [None] * _no
_cleanedo = [None] * _no

for _bs in tqdm(range(0, _no, 5), desc=f"OpenAI/{_model_oai}", unit="batch"):
    _be  = min(_bs + 5, _no)
    _raw = _call(_client_oai, _model_oai, _preo[_bs:_be], start_idx=_bs)
    for _li in range(_be - _bs):
        _gi = _bs + _li
        _m  = next((r for r in _raw if isinstance(r, dict) and r.get("idx") == _gi), None)
        if _m is None: _m = _raw[_li] if _li < len(_raw) and isinstance(_raw[_li], dict) else {}
        _c  = clean_llm_output(_m)
        _canono[_gi]   = build_canonical_name(_c)
        _cleanedo[_gi] = _c
    time.sleep(_sleep_oai)

_t2no = {_uniqo[i]: _canono[i] or ""   for i in range(_no)}
_t2co = {_uniqo[i]: _cleanedo[i] or {} for i in range(_no)}

_dfo["item_a_normalized"] = [_t2no.get(t, "") for t in _tao]
_dfo["item_b_normalized"] = [_t2no.get(t, "") for t in _tbo]
for _f in ["tipo_produto", "marca", "modelo", "capacidade_volume", "voltagem", "quantidade"]:
    _dfo[f"a_{_f}"] = [_t2co.get(t, {}).get(_f) for t in _tao]
    _dfo[f"b_{_f}"] = [_t2co.get(t, {}).get(_f) for t in _tbo]

_dfo.to_csv(_out_oai, index=False, encoding="utf-8-sig")
print(f"Salvo: {_out_oai}")
print("Benchmark: mude PAIRS_NORMALIZED_CSV =", repr(_out_oai))

OpenAI/gpt-5.4-mini:   0%|          | 0/452 [00:00<?, ?batch/s]

2026-08-22 03:24:51,439 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
OpenAI/gpt-5.4-mini: 100%|██████████| 452/452 [19:15<00:00,  2.56s/batch]

Salvo: dados/pairs_normalized_openai_gpt5.4mini_run3.csv
Benchmark: mude PAIRS_NORMALIZED_CSV = 'dados/pairs_normalized_openai_gpt5.4mini_run3.csv'


In [167]:
_dfo

,title_a,title_b,candidate_label,similarity_score,label,item_a_normalized,item_b_normalized,a_tipo_produto,b_tipo_produto,a_marca,b_marca,a_modelo,b_modelo,a_capacidade_volume,b_capacidade_volume,a_voltagem,b_voltagem,a_quantidade,b_quantidade
0,Pulseira Casio Original Prata Gshock Dw5600e,Relógio Bvulgari,non_match,0.1967,0,pulseira Casio prata dw5600e,,pulseira,None,Casio,None,dw5600e,None,None,None,None,None,None,None
1,Trimpot 3386f De 500k Uma Volta Lote Com 25 Peças,Potenciometro Do A16o A190,non_match,0.1867,0,,,None,None,None,None,None,None,None,None,None,None,None,None
2,Relógio De Parede Para Cozinha Talher Colher P...,Relógio Parede Personalizado Lata Pokemon,non_match,0.4894,0,,,None,None,None,None,None,None,None,None,None,None,None,None
3,Mesa Em Madeira Rústica-pubs-bar- Johnnie Walk...,Frete Grátis | Cerveja 55 Sem Glúten | 18 Lata...,non_match,0.2478,0,,cerveja - 18 710 ml,None,cerveja,None,None,None,None,None,710ml,None,None,None,18
4,Cadeirinha Reclinável Preta Maestro Bebê 0 À 2...,Chicco - Xpace - Cadeirinha P/ Auto - Excelent...,non_match,0.3667,0,,,None,None,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,Cd Luis Miguel - Romances,Dvd - Eric Clapton - Wonderful Tonight Japan 2...,non_match,0.2143,0,,,None,None,None,None,None,None,None,None,None,None,None,None
1196,Gps Automotivo March Todos Tela 4.3 Voz Tv Dig...,Gps Automotivo Vw Santana Tela 4.3 Voz Tv Digi...,match,0.8571,0,,gps automotivo Vw santana,None,gps automotivo,None,Vw,None,santana,None,None,None,None,None,None
1197,Reservatório De Óleo De Direção Hidráulica Cap...,Reservatório De Óleo Direção Hidráulica Fiat D...,match,0.7928,0,reservatório óleo v6 6c,,reservatório óleo,None,None,None,v6 6c,None,None,None,None,None,None,None
1198,"Megafone Portátil Com Gravação, Memoria E Musi...",Patinete Disney Frozen - Bandeirante,random,0.2045,0,megafone 35w,patinete Bandeirante,megafone,patinete,None,Bandeirante,None,None,None,None,None,None,None,None
